# SQL Interview Practice — 25 Questions
### Ordered: Easy → Medium → Hard
Write your SQL in the **Your Answer** cell, then run the **Solution** cell to verify.

In [ ]:
!pip install duckdb pandas -q
import duckdb
import pandas as pd

swap_rows = pd.DataFrame({'order_id': [1,2,3,4,5], 'item': ['A','B','C','D','E']})

employee = pd.DataFrame({
    'id':         [1,2,3,4,5,6],
    'name':       ['Alice','Bob','Charlie','Diana','Eve','Frank'],
    'dept':       ['Eng','Eng','HR','HR','Sales','Sales'],
    'salary':     [90000,90000,60000,75000,50000,50000],
    'manager_id': [None,1,None,3,None,5]
})

sales = pd.DataFrame({
    'sale_date':  pd.to_datetime(['2024-01-01','2024-01-02','2024-01-03','2024-01-04','2024-01-05','2024-01-06','2024-01-07']),
    'category':   ['A','A','A','B','B','B','A'],
    'amount':     [100,200,150,300,100,250,80]
})

logins = pd.DataFrame({
    'user_id':    [1,1,1,1,2,2,3,3,3,3,3],
    'login_date': pd.to_datetime(['2024-01-01','2024-01-02','2024-01-04','2024-01-05',
                                  '2024-01-01','2024-01-02',
                                  '2024-01-01','2024-01-02','2024-01-03','2024-01-01','2024-01-02'])
})

customers = pd.DataFrame({'customer_id': [101,102,103,104,105], 'name': ['Alice','Bob','Charlie','Diana','Eve']})

orders = pd.DataFrame({
    'order_id':    [1,2,3,4,5],
    'customer_id': [101,102,101,103,102],
    'item':        ['A','B','C','D','E'],
    'order_date':  pd.to_datetime(['2024-01-05','2024-01-03','2024-01-01','2024-01-07','2024-01-02']),
    'amount':      [250,180,320,90,150]
})

products = pd.DataFrame({'product_id': [1,2,3], 'product_name': ['Laptop','Mouse','Keyboard'], 'price': [1200,25,80]})
order_items = pd.DataFrame({'order_id': [1,1,2,3,3,4], 'product_id': [1,2,2,3,1,2], 'qty': [1,2,3,1,1,5]})

attendance = pd.DataFrame({
    'att_date': pd.to_datetime(['2024-01-01','2024-01-02','2024-01-03','2024-01-04','2024-01-05','2024-01-06']),
    'status':   ['present','present','absent','present','present','absent']
})

users = pd.DataFrame({
    'user_id': [1,2,3,4,5,6],
    'email':   ['a@x.com','b@x.com','a@x.com','c@x.com','b@x.com','d@x.com'],
    'name':    ['Alice','Bob','Alex','Charlie','Betty','Dave']
})

revenue = pd.DataFrame({
    'rev_date': pd.to_datetime(['2024-01-01','2024-01-02','2024-01-03','2024-01-04','2024-01-05','2024-01-06']),
    'revenue':  [100,200,150,300,250,180]
})

transactions = pd.DataFrame({
    'txn_id':      [1,2,3,4,5,6,7],
    'customer_id': [101,101,102,102,103,103,101],
    'txn_date':    pd.to_datetime(['2024-01-01','2024-02-15','2024-01-10','2024-03-01','2024-01-05','2024-01-05','2024-03-10']),
    'amount':      [500,300,200,450,100,100,700]
})

dept_budget = pd.DataFrame({'dept': ['Eng','HR','Sales'], 'budget': [500000,200000,300000]})

scores = pd.DataFrame({
    'student_id': [1,2,3,4,5],
    'name':       ['Alice','Bob','Carol','Dave','Eve'],
    'score':      [85, None, 90, None, 78]
})

inventory = pd.DataFrame({
    'product_id':   [1,2,3,4,5],
    'product_name': ['Laptop','Mouse','Keyboard','Monitor','Webcam'],
    'stock':        [5,0,12,0,3],
    'price':        [1200,25,80,400,60]
})

print('All DataFrames ready!')

---
## ⭐ EASY
---
## Q1: Stock Status Label
Label each product as `Out of Stock` (stock=0), `Low Stock` (1–5), or `In Stock` (>5).

DataFrame: `inventory` — columns: `product_id`, `product_name`, `stock`, `price`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT product_name, stock, price,
    CASE
        WHEN stock = 0  THEN 'Out of Stock'
        WHEN stock <= 5 THEN 'Low Stock'
        ELSE                 'In Stock'
    END AS stock_status
FROM inventory
ORDER BY stock
""").df()

---
## Q2: NULL Handling — COUNT vs COUNT(*)
Return: total rows, number of students who have a score, and the correct average score.

DataFrame: `scores` — columns: `student_id`, `name`, `score` (some scores are NULL)

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    COUNT(*)            AS total_rows,
    COUNT(score)        AS students_with_score,
    ROUND(AVG(score),2) AS correct_avg,
    ROUND(SUM(score) * 1.0 / COUNT(*), 2) AS wrong_avg_if_null_treated_as_zero
FROM scores
""").df()

---
## Q3: Filter Departments by Average Salary
Return departments where the average salary is greater than 70000.

DataFrame: `employee` — columns: `dept`, `salary`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT dept, ROUND(AVG(salary), 2) AS avg_salary
FROM employee
GROUP BY dept
HAVING AVG(salary) > 70000
ORDER BY avg_salary DESC
""").df()

---
## Q4: Find Duplicate Emails — Return Full Rows
Find all rows where the `email` appears more than once. Return the full row, not just the email.

DataFrame: `users` — columns: `user_id`, `email`, `name`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT user_id, email, name
FROM (
    SELECT *, COUNT(*) OVER (PARTITION BY email) AS cnt
    FROM users
) t
WHERE cnt > 1
ORDER BY email, user_id
""").df()

---
## Q5: UNION vs UNION ALL
Combine employee names from `Eng` and `HR` departments — once with UNION, once with UNION ALL.
Run both and observe the row count difference.

DataFrame: `employee`

In [ ]:
# Your Answer — UNION
duckdb.sql("""

""").df()

In [ ]:
# Your Answer — UNION ALL
duckdb.sql("""

""").df()

In [ ]:
# Solution
print('--- UNION (deduplicates) ---')
display(duckdb.sql("""
    SELECT name FROM employee WHERE dept = 'Eng'
    UNION
    SELECT name FROM employee WHERE dept = 'HR'
""").df())
print('--- UNION ALL (keeps all rows) ---')
display(duckdb.sql("""
    SELECT name FROM employee WHERE dept = 'Eng'
    UNION ALL
    SELECT name FROM employee WHERE dept = 'HR'
""").df())

---
## Q6: Customers Who Never Ordered
Return customers who have no rows in `orders`.

DataFrames: `customers` (customer_id, name), `orders` (order_id, customer_id, ...)

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT c.customer_id, c.name
FROM customers c
WHERE NOT EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
)
""").df()

---
## Q7: Revenue Per Product Per Order
For each order, show each product's line revenue (qty * price).

DataFrames: `order_items` (order_id, product_id, qty), `products` (product_id, product_name, price)

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    oi.order_id,
    p.product_name,
    oi.qty,
    p.price,
    oi.qty * p.price AS line_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
ORDER BY oi.order_id, line_revenue DESC
""").df()

---
## Q8: All Customers With Total Order Amount
Return all customers with their total order amount. Customers with no orders should show `0`, not NULL.

DataFrames: `customers` (customer_id, name), `orders` (order_id, customer_id, amount)

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT c.customer_id, c.name,
    COALESCE(SUM(o.amount), 0) AS total_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id
""").df()

---
## ⭐⭐ MEDIUM
---
## Q9: Employees Earning More Than Their Manager
Find employees whose salary is higher than their direct manager's salary.

DataFrame: `employee` — columns: `id`, `name`, `salary`, `manager_id`

In [ ]:
# Preview
duckdb.sql("SELECT * FROM employee").df()

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT e.name AS employee, e.salary, m.name AS manager, m.salary AS manager_salary
FROM employee e
JOIN employee m ON e.manager_id = m.id
WHERE e.salary > m.salary
""").df()

---
## Q10: Second Highest Salary
Return the second highest **distinct** salary. Do not use LIMIT/OFFSET.

DataFrame: `employee` — columns: `id`, `name`, `dept`, `salary`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT MAX(salary) AS second_highest_salary
FROM (
    SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM employee
) ranked
WHERE rnk = 2
""").df()

---
## Q11: Highest Paid Employee Per Department
Return all employees with the highest salary in their department. Include ties.

DataFrame: `employee` — columns: `id`, `name`, `dept`, `salary`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT id, name, dept, salary
FROM (
    SELECT *, RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk
    FROM employee
) t
WHERE rnk = 1
""").df()

---
## Q12: ROW_NUMBER vs RANK vs DENSE_RANK
Show salary rank per department using all three ranking functions side by side.

DataFrame: `employee` — columns: `name`, `dept`, `salary`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    name, dept, salary,
    ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS row_num,
    RANK()       OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk,
    DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dense_rnk
FROM employee
ORDER BY dept, salary DESC
""").df()

---
## Q13: Day-over-Day Revenue Change
For each day, show the revenue, the previous day's revenue, and the difference.

DataFrame: `revenue` — columns: `rev_date`, `revenue`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    rev_date, revenue,
    LAG(revenue) OVER (ORDER BY rev_date) AS prev_revenue,
    revenue - COALESCE(LAG(revenue) OVER (ORDER BY rev_date), 0) AS daily_change
FROM revenue
ORDER BY rev_date
""").df()

---
## Q14: 3-Day Moving Average of Revenue
Calculate the 3-day moving average of `revenue` ordered by `rev_date`.

DataFrame: `revenue` — columns: `rev_date`, `revenue`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    rev_date, revenue,
    ROUND(AVG(revenue) OVER (
        ORDER BY rev_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_3day
FROM revenue
ORDER BY rev_date
""").df()

---
## Q15: Running Total Reset By Category
Calculate a running total of `amount` per `category`, ordered by `sale_date`.

DataFrame: `sales` — columns: `sale_date`, `category`, `amount`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    sale_date, category, amount,
    SUM(amount) OVER (
        PARTITION BY category
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM sales
ORDER BY category, sale_date
""").df()

---
## Q16: Conditional Aggregation
For each category in `sales`, show:
- Total amount across all dates
- Total amount only for dates after 2024-01-03
- Count of distinct sale dates

DataFrame: `sales` — columns: `sale_date`, `category`, `amount`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    category,
    SUM(amount) AS total_amount,
    SUM(amount) FILTER (WHERE sale_date > '2024-01-03') AS amount_after_jan3,
    COUNT(DISTINCT sale_date) AS num_days
FROM sales
GROUP BY category
ORDER BY category
""").df()

---
## Q17: Salary as % of Department Budget
For each employee, calculate their salary as a percentage of their department's total budget.

DataFrames: `employee` (name, dept, salary), `dept_budget` (dept, budget)

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    e.name, e.dept, e.salary, d.budget,
    ROUND(e.salary * 100.0 / d.budget, 2) AS salary_pct_of_budget
FROM employee e
JOIN dept_budget d ON e.dept = d.dept
ORDER BY e.dept, salary_pct_of_budget DESC
""").df()

---
## ⭐⭐⭐ HARD
---
## Q18: First Order Per Customer
Return the full row of each customer's earliest order. If two orders share the same date, pick the lower `order_id`.

DataFrame: `orders` — columns: `order_id`, `customer_id`, `item`, `order_date`, `amount`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT order_id, customer_id, item, order_date, amount
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS rn
    FROM orders
) t
WHERE rn = 1
""").df()

---
## Q19: Cumulative % of Total Revenue
For each day, show the revenue, cumulative revenue so far, and cumulative revenue as a % of the overall total.

DataFrame: `revenue` — columns: `rev_date`, `revenue`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    rev_date, revenue,
    SUM(revenue) OVER (ORDER BY rev_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_revenue,
    ROUND(
        SUM(revenue) OVER (ORDER BY rev_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
        * 100.0 / SUM(revenue) OVER (), 2
    ) AS cumulative_pct
FROM revenue
ORDER BY rev_date
""").df()

---
## Q20: Customer Retention — Month-over-Month
Find customers who made a transaction in both January 2024 and February 2024.

DataFrame: `transactions` — columns: `txn_id`, `customer_id`, `txn_date`, `amount`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT DISTINCT t1.customer_id
FROM transactions t1
JOIN transactions t2 ON t1.customer_id = t2.customer_id
WHERE DATE_TRUNC('month', t1.txn_date) = '2024-01-01'
  AND DATE_TRUNC('month', t2.txn_date) = '2024-02-01'
""").df()

---
## Q21: Label Customers by Spend Band
Label each customer as `High` (top 25%), `Mid` (middle 50%), or `Low` (bottom 25%) based on total transaction amount.

DataFrame: `transactions` — columns: `customer_id`, `amount`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
WITH totals AS (
    SELECT customer_id, SUM(amount) AS total_spend
    FROM transactions GROUP BY customer_id
),
ranked AS (
    SELECT customer_id, total_spend,
        NTILE(4) OVER (ORDER BY total_spend) AS quartile
    FROM totals
)
SELECT customer_id, total_spend,
    CASE
        WHEN quartile = 4 THEN 'High'
        WHEN quartile IN (2,3) THEN 'Mid'
        ELSE 'Low'
    END AS spend_band
FROM ranked
ORDER BY total_spend DESC
""").df()

---
## Q22: Employees Without Subordinates (Leaf Nodes)
Find employees whose `id` never appears as anyone else's `manager_id`.

DataFrame: `employee` — columns: `id`, `name`, `dept`, `manager_id`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT id, name, dept
FROM employee e
WHERE NOT EXISTS (
    SELECT 1 FROM employee sub WHERE sub.manager_id = e.id
)
""").df()

---
## Q23: Gaps and Islands — Continuous Date Ranges
Group consecutive days with the same `status` into ranges.
Output columns: `start_date`, `end_date`, `status`.

DataFrame: `attendance` — columns: `att_date`, `status`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
WITH grouped AS (
    SELECT att_date, status,
        att_date - INTERVAL (ROW_NUMBER() OVER (PARTITION BY status ORDER BY att_date)) DAY AS grp
    FROM attendance
)
SELECT MIN(att_date) AS start_date, MAX(att_date) AS end_date, status
FROM grouped
GROUP BY status, grp
ORDER BY start_date
""").df()

---
## Q24: Find Users With 3+ Consecutive Login Days
Find all users who logged in for at least 3 consecutive days.
Note: the table has duplicate login dates per user — handle them.

DataFrame: `logins` — columns: `user_id`, `login_date`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
WITH deduped AS (
    SELECT DISTINCT user_id, login_date FROM logins
),
ranked AS (
    SELECT user_id, login_date,
        login_date - INTERVAL (ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date)) DAY AS grp
    FROM deduped
),
streaks AS (
    SELECT user_id, grp, COUNT(*) AS streak_len
    FROM ranked GROUP BY user_id, grp
)
SELECT DISTINCT user_id FROM streaks WHERE streak_len >= 3
""").df()

---
## Q25: Swap Adjacent Rows
Swap each pair of adjacent rows by `order_id`. If the last row is odd-numbered, keep it in place.

DataFrame: `swap_rows` — columns: `order_id`, `item`

In [ ]:
# Your Answer
duckdb.sql("""

""").df()

In [ ]:
# Solution
duckdb.sql("""
SELECT
    order_id,
    CASE
        WHEN order_id % 2 = 1 AND LEAD(order_id) OVER (ORDER BY order_id) IS NOT NULL
            THEN LEAD(item) OVER (ORDER BY order_id)
        WHEN order_id % 2 = 0
            THEN LAG(item)  OVER (ORDER BY order_id)
        ELSE item
    END AS item
FROM swap_rows
ORDER BY order_id
""").df()